# 스프린트 미션 15 · 연구자 2 — 추론 (Inference)

연구자 2 는 사전에 데이터·모델을 보유하지 않는다.  
연구자 1 이 Docker Hub 에 올린 이미지를 `docker-compose` 로 실행하면,
연구자 1 컨테이너가 생성한 **model.pkl** 과 **test.csv** 가 공유 볼륨을 통해
이 Jupyter 컨테이너의 `/home/jovyan/shared` 경로로 전달된다.

이 노트북은 그 두 파일을 읽어 추론을 수행하고 결과를 `result.csv` 로 저장한다.

## 1. 라이브러리 및 경로 설정

In [1]:
import os
import pandas as pd
import joblib

SHARED_DIR = "/home/jovyan/shared"   # 연구자 1 이 넘겨준 파일 (공유 볼륨)
WORK_DIR   = "/home/jovyan/work"     # 결과 저장 (호스트에 보존)

MODEL_PATH  = os.path.join(SHARED_DIR, "model.pkl")
TEST_PATH   = os.path.join(SHARED_DIR, "mission15_test.csv")
RESULT_PATH = os.path.join(WORK_DIR, "result.csv")

print("전달받은 파일 목록:", os.listdir(SHARED_DIR))

전달받은 파일 목록: ['mission15_test.csv', 'model.pkl']


## 2. 모델과 테스트 데이터 로드
`model.pkl` 은 전처리(범주형 인코딩)까지 포함된 Pipeline 이므로 원본 test.csv 를 그대로 넣을 수 있다.  
※ 연구자 1 과 scikit-learn 버전이 동일해야 경고 없이 로드된다 (requirements.txt 통일).

In [2]:
model = joblib.load(MODEL_PATH)
test_df = pd.read_csv(TEST_PATH)
print("모델 로드 완료:", type(model).__name__)
print("test shape:", test_df.shape)
test_df.head()

모델 로드 완료: Pipeline
test shape: (3000, 5)


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced
0,7,99,Yes,9,1
1,8,51,Yes,7,2
2,8,91,No,4,5
3,5,79,No,7,8
4,2,72,No,4,3


## 3. 추론 수행

In [4]:
preds = model.predict(test_df)
print("예측 개수:", len(preds))
print("예측 예시:", preds[:5].round(2))

예측 개수: 3000
예측 예시: [91.85 45.07 84.33 65.58 47.43]


## 4. 결과를 result.csv 로 저장
원본 피처에 예측 컬럼 `Predicted Performance Index` 를 붙여 저장한다.  
성취도 지수는 가장 가까운 정수로 반올림되는 지표이므로 반올림 값도 함께 제공한다.

In [5]:
result = test_df.copy()
result["Predicted Performance Index"] = preds
result["Predicted Performance Index (rounded)"] = preds.round().astype(int)
result.to_csv(RESULT_PATH, index=False)
print("저장 완료:", RESULT_PATH)
result.head()

저장 완료: /home/jovyan/work/result.csv


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Predicted Performance Index,Predicted Performance Index (rounded)
0,7,99,Yes,9,1,91.849258,92
1,8,51,Yes,7,2,45.070334,45
2,8,91,No,4,5,84.331772,84
3,5,79,No,7,8,65.584878,66
4,2,72,No,4,3,47.428984,47


## 5. 정리
- 연구자 1 의 Docker Hub 이미지 → 컨테이너 실행 → `model.pkl`, `test.csv` 생성
- 공유 볼륨(`shared-data`)을 통해 이 Jupyter 컨테이너로 자동 전달
- 전처리 포함 Pipeline 이라 원본 test.csv 를 그대로 추론
- 결과를 `result.csv` 로 저장 (호스트 `researcher2/work/` 에 영구 보존)